In [1]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


%load_ext sparksql_magic

In [2]:
# 1. Configure Spark Session for Delta Lake
builder = SparkSession.builder \
    .appName("DeltaLakeLocal") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Automatically downloads the correct matching Delta jars
spark = configure_spark_with_delta_pip(builder).getOrCreate()
%config SparkSqlMagic.spark = spark

your 131072x1 screen size is bogus. expect trouble
26/05/18 10:44:45 WARN Utils: Your hostname, DESKTOP-IS86RQE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/18 10:44:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/obaliuta/miniconda3/envs/spark_with_delta/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/obaliuta/.ivy2/cache
The jars for the packages stored in: /home/obaliuta/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a11e6a1d-35a2-4215-8b95-99ee6928f137;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 286ms :: artifacts dl 11ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |

In [3]:
# 2. Create sample data
data = [("Alice", 34), ("Bob", 45), ("Charlie", 29)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)

In [4]:
# 3. Save DataFrame as a Delta Table using the absolute home path
home_dir = os.path.expanduser("~")
delta_path = os.path.join(home_dir, "lakehouse_test/test_init_table")

df.write.format("delta").mode("overwrite").save(delta_path)
print(f"Delta table created successfully at: {delta_path}")

26/05/18 10:45:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/05/18 10:45:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta table created successfully at: /home/obaliuta/lakehouse_test/test_init_table


In [5]:
df_sql_path = spark.sql(f"SELECT * FROM delta.`{delta_path}`")

In [6]:
%%sparksql
SELECT * FROM delta.`/home/obaliuta/lakehouse_test/test_init_table`

Name,Age
Charlie,29
Alice,34
Bob,45


In [7]:
spark.stop()